In [ ]:
%pip install faiss-cpu sentence-transformers mlflow pypdf --quiet
dbutils.library.restartPython()

In [0]:
import os
import json
import pickle
import mlflow
import faiss
import numpy as np
from typing import List, Tuple

from sentence_transformers import SentenceTransformer
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document

CONFIG_PATH = "/Volumes/main/default/dino_pdfs/project/config/project_config.json"
with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

ARTIFACTS_URI = config["artifacts_uri"]
INDEX_URI = config["index_uri"]
EMBEDDING_MODEL_NAME = config["embedding_model_name"]

mlflow.set_experiment(config["mlflow_experiment"])

print("Config loaded successfully.")

Config loaded successfully.


In [0]:
chunks_pickle_path = os.path.join(ARTIFACTS_URI, "chunked_docs.pkl")

try:
    with open(chunks_pickle_path, "rb") as f:
        chunked_docs = pickle.load(f)
    print("Loaded chunks:", len(chunked_docs))
except (KeyError, AttributeError) as e:
    print(f"Pydantic version mismatch detected: {e}")
    print("Reconstructing documents from the original source...")
    
    # Reload from the previous notebook's output
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    # Load config
    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)
    
    VOLUME_URI = config["volume_uri"]
    CHUNK_SIZE = int(config["chunk_size"])
    CHUNK_OVERLAP = int(config["chunk_overlap"])
    
    # Load PDFs
    all_docs = []
    for f in dbutils.fs.ls(VOLUME_URI):
        if not f.isDir() and f.name.lower().endswith(".pdf"):
            volume_path = f.path.replace('dbfs:', '')
            loader = PyPDFLoader(volume_path)
            docs = loader.load()
            for d in docs:
                d.metadata["source_file"] = f.name
                d.metadata["source_path"] = f.path
                d.metadata["domain"] = "dinosaurs"
            all_docs.extend(docs)
    
    print(f"Loaded {len(all_docs)} pages")
    
    # Chunk documents
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len
    )
    chunked_docs = text_splitter.split_documents(all_docs)
    
    # Add chunk IDs
    for i, chunk in enumerate(chunked_docs):
        chunk.metadata["chunk_id"] = i
    
    print(f"Created {len(chunked_docs)} chunks")
    
    # Save updated pickle
    with open(chunks_pickle_path, "wb") as f:
        pickle.dump(chunked_docs, f)
    print(f"Saved updated chunks to: {chunks_pickle_path}")

Pydantic version mismatch detected: '__fields_set__'
Reconstructing documents from the original source...
Loaded 47 pages
Created 167 chunks
Saved updated chunks to: /Volumes/main/default/dino_pdfs/project/artifacts/chunked_docs.pkl


In [0]:
class SBERTEmbeddings(Embeddings):
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts):
        return self.model.encode(texts, show_progress_bar=False).tolist()

    def embed_query(self, text):
        return self.model.encode([text], show_progress_bar=False)[0].tolist()

embeddings = SBERTEmbeddings(EMBEDDING_MODEL_NAME)
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [0]:
# FAISS vector store
class SimpleFAISSVectorStore:
    def __init__(self, embeddings_model, documents=None):
        self.embeddings_model = embeddings_model
        self.documents = documents or []
        self.index = None
        
        if documents:
            self._build_index()
    
    def _build_index(self):
        texts = [doc.page_content for doc in self.documents]
        embeddings = self.embeddings_model.embed_documents(texts)
        embeddings_array = np.array(embeddings).astype('float32')
        
        dimension = embeddings_array.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings_array)
        print(f"Built FAISS index with {len(self.documents)} documents")
    
    def save_local(self, folder_path):
        os.makedirs(folder_path, exist_ok=True)
        faiss.write_index(self.index, os.path.join(folder_path, "index.faiss"))
        
        with open(os.path.join(folder_path, "documents.pkl"), "wb") as f:
            pickle.dump(self.documents, f)
        print(f"Saved FAISS index to: {folder_path}")
    
    @classmethod
    def load_local(cls, folder_path, embeddings_model):
        index = faiss.read_index(os.path.join(folder_path, "index.faiss"))
        
        with open(os.path.join(folder_path, "documents.pkl"), "rb") as f:
            documents = pickle.load(f)
        
        vs = cls(embeddings_model)
        vs.index = index
        vs.documents = documents
        return vs
    
    def similarity_search(self, query: str, k: int = 4):
        query_embedding = self.embeddings_model.embed_query(query)
        query_array = np.array([query_embedding]).astype('float32')
        
        distances, indices = self.index.search(query_array, k)
        
        results = []
        for idx in indices[0]:
            if idx < len(self.documents):
                results.append(self.documents[idx])
        
        return results
    
    def as_retriever(self, search_kwargs=None):
        search_kwargs = search_kwargs or {}
        k = search_kwargs.get("k", 4)
        
        class Retriever:
            def __init__(self, vectorstore, k):
                self.vectorstore = vectorstore
                self.k = k
            
            def invoke(self, query):
                return self.vectorstore.similarity_search(query, self.k)
        
        return Retriever(self, k)

vectorstore = SimpleFAISSVectorStore(embeddings, chunked_docs)
vectorstore.save_local(INDEX_URI)

print("Saved FAISS index to:", INDEX_URI)

Built FAISS index with 167 documents
Saved FAISS index to: /Volumes/main/default/dino_pdfs/project/faiss_index
Saved FAISS index to: /Volumes/main/default/dino_pdfs/project/faiss_index


In [0]:
retriever = vectorstore.as_retriever(search_kwargs={"k": int(config["top_k"])})
query = "What did Triceratops eat?"
docs = retriever.invoke(query)

for i, d in enumerate(docs, start=1):
    print(f"\nResult {i}")
    print("Source:", d.metadata.get("source_file"))
    print("Page:", d.metadata.get("page"))
    print("Chunk ID:", d.metadata.get("chunk_id"))
    print(d.page_content[:300])

Result 1
Source: Document3_Dinosaur_Classification_Size_and_Behavior.pdf
Page: 4
Chunk ID: 75
other. Additional evidence for attacking live prey is the partially healed tail of 
an Edmontosaurus, a hadrosaurid dinosaur; the tail is damaged in such a way that 
shows the animal was bitten by a tyrannosaur but survived. Cannibalism amongst 
some species of dinosaurs was confirmed by tooth marks
Result 2
Source: Document2_History_of_Dinosaur_Discovery.pdf
Page: 4
Chunk ID: 56
indications of diet and digestive system anatomy (e.g., crops).[87][88] More 
concrete evidence of internal anatomy has been reported in Scipionyx from 
the Pietraroja Plattenkalk of Italy. It preserves portions of the intestines, colon, 
liver, muscles, and windpipe.[89] 
REFERENCE: 
Wikipedia.
Result 3
Source: Document3_Dinosaur_Classification_Size_and_Behavior.pdf
Page: 4
Chunk ID: 74
depicting carnivorous theropods as pack hunters working together to bring down 
large prey. However, this lifestyle is uncommon amon

In [0]:
with mlflow.start_run(run_name="02_embeddings_and_vector_store"):
    mlflow.log_param("embedding_model_name", EMBEDDING_MODEL_NAME)
    mlflow.log_param("vector_store", "FAISS")
    mlflow.log_metric("num_chunks_indexed", len(chunked_docs))
    mlflow.log_param("default_top_k", int(config["top_k"]))

print("Embedding/index run logged to MLflow.")

Embedding/index run logged to MLflow.
